### 거래량 데이터

`-` 행정표준코드의 법정동 코드를 활용하여 전국 시군구(법정동 코드 앞 5자리)를 추출하고, 국토교통부 아파트 매매 실거래 API를 통해 2020년 1월부터 2026년 6월까지의 거래 데이터를 자동 수집하였다.

`-` 수집된 실거래 데이터를 월별·시도별로 집계하여 거래량을 생성하고, 이후 거시경제 지표와 결합할 수 있는 분석용 데이터셋을 구축하였다.

- [법정동 코드](https://www.code.go.kr/stdcode/normalCodeL.do?utm_source=chatgpt.com)
- [국토교통부_아파트매매 실거래 상세 자료](https://www.data.go.kr/iim/api/selectAPIAcountView.do)

In [3]:
import pandas as pd

df = pd.read_csv(
    "/Users/CHOIEUNSEO/Desktop/AI·AX 전문가 과정/프로젝트/data/법정동코드 전체자료.txt",
    sep="\t",
    encoding="cp949"
)

df.head()

,법정동코드,법정동명,폐지여부
0,1100000000,서울특별시,존재
1,1111000000,서울특별시 종로구,존재
2,1111010100,서울특별시 종로구 청운동,존재
3,1111010200,서울특별시 종로구 신교동,존재
4,1111010300,서울특별시 종로구 궁정동,존재


In [4]:
# 현재 존재하는 행정구역만 사용
df = df[df["폐지여부"] == "존재"].copy()

# 시군구 코드(앞 5자리)
df["LAWD_CD"] = df["법정동코드"].astype(str).str[:5]

# 시도 / 시군구 분리
split_name = df["법정동명"].str.split()

df["시도"] = split_name.str[0]

# 시군구명 생성
def make_sigungu(x):
    if len(x) >= 3 and (x[1].endswith("시") and x[2].endswith(("구", "군"))):
        # 예) 경기도 성남시 분당구
        return x[1] + " " + x[2]
    elif len(x) >= 2:
        # 예) 서울특별시 강남구, 전북특별자치도 고창군
        return x[1]
    return ""

df["시군구"] = split_name.apply(make_sigungu)

# 시군구별 중복 제거
lawd = (
    df[["시도", "시군구", "LAWD_CD"]]
    .drop_duplicates(subset="LAWD_CD")
    .sort_values(["시도", "LAWD_CD"])
    .reset_index(drop=True)
)

In [5]:
lawd = lawd[lawd["시도"] == "서울특별시"].reset_index(drop=True)

# 서울특별시 전체 코드 지우기. 
# 구별 거래량을 모아서 서울 전체 거래량을 만들 거라서
lawd = lawd[lawd["시군구"] != ""].reset_index(drop=True)

In [6]:
lawd[lawd["시도"] == '서울특별시']

,시도,시군구,LAWD_CD
0,서울특별시,종로구,11110
1,서울특별시,중구,11140
2,서울특별시,용산구,11170
3,서울특별시,성동구,11200
4,서울특별시,광진구,11215
5,서울특별시,동대문구,11230
6,서울특별시,중랑구,11260
7,서울특별시,성북구,11290
8,서울특별시,강북구,11305
9,서울특별시,도봉구,11320


In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

serviceKey = os.getenv("OPENAI_API_KEY")

In [8]:
import os
import time
import requests
import xmltodict
import pandas as pd
from tqdm import tqdm

##############################################
# 설정
##############################################

SERVICE_KEY = requests.utils.unquote(serviceKey)

URL = "https://apis.data.go.kr/1613000/RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade"

SAVE_DIR = "raw"

os.makedirs(SAVE_DIR, exist_ok=True)

##############################################
# 월 생성
##############################################

months = pd.date_range(
    "2020-01-01",
    "2026-06-01",
    freq="MS"
).strftime("%Y%m")

##############################################
# lawd 사용
##############################################
# lawd 컬럼
#
# 시도
# 시군구
# LAWD_CD
#
##############################################

def request_page(code, month, page):

    params = {
        "serviceKey": SERVICE_KEY,
        "LAWD_CD": code,
        "DEAL_YMD": month,
        "pageNo": page,
        "numOfRows": 999
    }

    r = requests.get(URL, params=params, timeout=30)

    data = xmltodict.parse(r.text)

    body = data["response"]["body"]

    total = int(body.get("totalCount", 0))

    items = body["items"]

    if items is None:
        return total, []

    items = items.get("item")

    if items is None:
        return total, []

    if isinstance(items, dict):
        items = [items]

    return total, items

##############################################
# 월별 수집
##############################################

for month in months:

    save_path = f"{SAVE_DIR}/{month}.csv"

    if os.path.exists(save_path):
        print(month, "이미 존재")
        continue

    result = []

    print(f"\n====== {month} ======")

    for _, row in tqdm(lawd.iterrows(), total=len(lawd)):

        code = row["LAWD_CD"]

        sido = row["시도"]

        sigungu = row["시군구"]

        page = 1

        while True:

            try:

                total, items = request_page(code, month, page)

                if len(items) == 0:
                    break

                for item in items:

                    item["시도"] = sido
                    item["시군구"] = sigungu

                    result.append(item)

                if page * 999 >= total:
                    break

                page += 1

                time.sleep(0.2)

            except Exception as e:

                print(code, month, e)

                break

    df_month = pd.DataFrame(result)

    df_month.to_csv(
        save_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(month, "저장 완료", len(df_month))
    

202001 이미 존재
202002 이미 존재
202003 이미 존재
202004 이미 존재
202005 이미 존재
202006 이미 존재
202007 이미 존재
202008 이미 존재
202009 이미 존재
202010 이미 존재
202011 이미 존재
202012 이미 존재
202101 이미 존재
202102 이미 존재
202103 이미 존재
202104 이미 존재
202105 이미 존재
202106 이미 존재
202107 이미 존재
202108 이미 존재
202109 이미 존재
202110 이미 존재
202111 이미 존재
202112 이미 존재
202201 이미 존재
202202 이미 존재
202203 이미 존재
202204 이미 존재
202205 이미 존재
202206 이미 존재
202207 이미 존재
202208 이미 존재
202209 이미 존재
202210 이미 존재
202211 이미 존재
202212 이미 존재
202301 이미 존재
202302 이미 존재
202303 이미 존재
202304 이미 존재
202305 이미 존재
202306 이미 존재
202307 이미 존재
202308 이미 존재
202309 이미 존재
202310 이미 존재
202311 이미 존재
202312 이미 존재
202401 이미 존재
202402 이미 존재
202403 이미 존재
202404 이미 존재
202405 이미 존재
202406 이미 존재
202407 이미 존재
202408 이미 존재
202409 이미 존재
202410 이미 존재
202411 이미 존재
202412 이미 존재
202501 이미 존재
202502 이미 존재
202503 이미 존재
202504 이미 존재
202505 이미 존재
202506 이미 존재
202507 이미 존재
202508 이미 존재
202509 이미 존재
202510 이미 존재
202511 이미 존재
202512 이미 존재
202601 이미 존재
202602 이미 존재
202603 이미 존재
202604 이미 존재
202605 이미 존재

In [9]:
import glob

files = sorted(glob.glob("raw/*.csv"))

df = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

df.to_csv(
    "apt_trade_all.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df.shape)

(356250, 22)


In [10]:
df

,aptDong,aptNm,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,dealDay,dealMonth,dealYear,...,excluUseAr,floor,jibun,landLeaseholdGbn,rgstDate,sggCd,slerGbn,umdNm,시도,시군구
0,NaN,창신쌍용1,1992,NaN,NaN,NaN,"31,000",3,1,2020,...,54.70,5,702,N,NaN,11110,NaN,창신동,서울특별시,종로구
1,NaN,삼성,1998,NaN,NaN,NaN,"53,000",13,1,2020,...,84.93,6,596,N,NaN,11110,NaN,평창동,서울특별시,종로구
2,NaN,광화문스페이스본(106동),2008,NaN,NaN,NaN,"162,000",2,1,2020,...,163.33,2,9-1,N,NaN,11110,NaN,사직동,서울특별시,종로구
3,NaN,한빛,1999,NaN,NaN,NaN,"34,500",10,1,2020,...,59.73,4,1-30,N,NaN,11110,NaN,명륜3가,서울특별시,종로구
4,NaN,창신쌍용1,1992,NaN,NaN,NaN,"68,500",19,1,2020,...,106.62,1,702,N,NaN,11110,NaN,창신동,서울특별시,종로구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356245,NaN,롯데캐슬퍼스트,2008,개인,NaN,NaN,"185,000",2,6,2026,...,84.98,23,414-2,N,NaN,11740,개인,암사동,서울특별시,강동구
356246,NaN,더샵파크솔레이유,2023,개인,NaN,NaN,"129,000",1,6,2026,...,53.24,2,654,N,NaN,11740,개인,둔촌동,서울특별시,강동구
356247,NaN,아남1,1996,개인,NaN,NaN,"153,500",1,6,2026,...,84.91,10,486,N,NaN,11740,개인,고덕동,서울특별시,강동구
356248,NaN,둔촌푸르지오,2010,개인,NaN,NaN,"149,500",1,6,2026,...,59.98,10,630,N,NaN,11740,개인,둔촌동,서울특별시,강동구


In [11]:
df = df[
    [
        "dealYear",
        "dealMonth",
        "dealAmount",
        "excluUseAr",
        "aptNm",
        "umdNm",
        "시도",
        "시군구"
    ]
]

df.columns = [
    "년",
    "월",
    "거래금액",
    "전용면적",
    "아파트명",
    "법정동",
    "시도",
    "시군구"
]

df["거래금액"] = (
    df["거래금액"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

trade_cnt = (
    df.groupby(["년","월","시도"])
      .size()
      .reset_index(name="거래량")
)

/var/folders/hf/5rb3dt8x3mq707hsrnthhx2w0000gn/T/ipykernel_62575/764015446.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["거래금액"] = (


In [12]:
trade_cnt.head()

,년,월,시도,거래량
0,2020,1,서울특별시,6500
1,2020,2,서울특별시,8403
2,2020,3,서울특별시,4634
3,2020,4,서울특별시,3165
4,2020,5,서울특별시,5819


In [13]:
trade_cnt.tail()

,년,월,시도,거래량
73,2026,2,서울특별시,5980
74,2026,3,서울특별시,5702
75,2026,4,서울특별시,8832
76,2026,5,서울특별시,8917
77,2026,6,서울특별시,3914


### 기준금리 데이터 

- [한국은행 기준금리 및 여수신금리](ecos.bok.or.kr/#/Short/193463)

In [14]:
base_rate = pd.read_csv("/Users/CHOIEUNSEO/Desktop/AI·AX 전문가 과정/프로젝트/data/기준금리.csv")

In [15]:
# 1. 날짜 컬럼들만 추출 ('/'가 포함된 컬럼명 찾기)
date_cols = [col for col in base_rate.columns if '/' in str(col)]

# 2. 기준금리 데이터가 있는 행(0번 행)만 선택해서 원본 보존하며 카피
df_target = base_rate.iloc[[0]].copy()

# 3. melt()를 사용하여 넓은 데이터를 길게 변환
df_melted = df_target.melt(
    value_vars=date_cols, 
    var_name='날짜', 
    value_name='금리'
)

# 4. '날짜' 컬럼(예: '2020/01')을 '년'과 '월'로 분리
df_melted['년'] = df_melted['날짜'].str.split('/').str[0].astype(int)
df_melted['월'] = df_melted['날짜'].str.split('/').str[1].astype(int)

# 5. 금리 데이터를 숫자형(float)으로 변환
df_melted['기준금리'] = df_melted['금리'].astype(float)

# 6. 최종적으로 원하는 컬럼만 선택하고 순서를 정렬하여 기존 base_rate 변수에 저장
base_rate = df_melted[['년', '월', '기준금리']]

In [16]:
base_rate.head()

,년,월,기준금리
0,2020,1,1.25
1,2020,2,1.25
2,2020,3,0.75
3,2020,4,0.75
4,2020,5,0.50


### 주택담보대출 금리 데이터

- [예금은행 대출금리(신규취급액 기준)_주택담보대출](ecos.bok.or.kr/#/Short/ef1310)

In [17]:
mortgage_rate = pd.read_csv("/Users/CHOIEUNSEO/Desktop/AI·AX 전문가 과정/프로젝트/data/주택담보대출금리.csv")

In [18]:
mortgage_rate

,통계표,계정항목,단위,변환,2020/01,2020/02,2020/03,2020/04,2020/05,2020/06,...,2025/08,2025/09,2025/10,2025/11,2025/12,2026/01,2026/02,2026/03,2026/04,2026/05
0,1.3.3.2.1. 예금은행 대출금리(신규취급액 기준),주택담보대출,연리%,원자료,2.51,2.52,2.48,2.58,2.52,2.49,...,3.96,3.96,3.98,4.17,4.23,4.29,4.32,4.34,4.31,4.32


In [19]:
# 1. 날짜 컬럼들만 추출 ('/'가 포함된 컬럼명 찾기)
date_cols = [col for col in mortgage_rate.columns if '/' in str(col)]

# 2. 주택담보대출 데이터가 있는 행(0번 행)만 선택해서 카피
df_target = mortgage_rate.iloc[[0]].copy()

# 3. melt()를 사용하여 넓은 데이터를 길게 변환
df_melted = df_target.melt(
    value_vars=date_cols, 
    var_name='날짜', 
    value_name='금리'
)

# 4. '날짜' 컬럼(예: '2020/01')을 '년'과 '월'로 분리
df_melted['년'] = df_melted['날짜'].str.split('/').str[0].astype(int)
df_melted['월'] = df_melted['날짜'].str.split('/').str[1].astype(int)

# 5. 금리 데이터를 숫자형(float)으로 변환
df_melted['주택담보대출금리'] = df_melted['금리'].astype(float)

# 6. 최종적으로 원하는 컬럼만 선택하고 순서 정렬
mortgage_rate = df_melted[['년', '월', '주택담보대출금리']]

In [20]:
mortgage_rate

,년,월,주택담보대출금리
0,2020,1,2.51
1,2020,2,2.52
2,2020,3,2.48
3,2020,4,2.58
4,2020,5,2.52
...,...,...,...
72,2026,1,4.29
73,2026,2,4.32
74,2026,3,4.34
75,2026,4,4.31


### 아파트 매매가격지수 데이터

- [한국부동산원 아파트 매매가격지수](https://www.reb.or.kr/r-one/portal/stat/easyStatPage/A_2024_00045.do?isMobileYn=N

In [32]:
hpi_df = pd.read_csv("/Users/CHOIEUNSEO/Desktop/AI·AX 전문가 과정/프로젝트/data/아파트_매매가격지수.csv", encoding='cp949')

In [33]:
hpi_df

,No,지역,지역.1,지역.2,지역.3,2020년 1월,2020년 1월.1,2020년 2월,2020년 2월.1,2020년 3월,...,2026년 1월,2026년 1월.1,2026년 2월,2026년 2월.1,2026년 3월,2026년 3월.1,2026년 4월,2026년 4월.1,2026년 5월,2026년 5월.1
0,No,지역,지역,지역,지역,지수,지수,지수,지수,지수,...,지수,지수,지수,지수,지수,지수,지수,지수,지수,지수
1,No,지역,지역,지역,지역,원자료,전기대비증감률,원자료,전기대비증감률,원자료,...,원자료,전기대비증감률,원자료,전기대비증감률,원자료,전기대비증감률,원자료,전기대비증감률,원자료,전기대비증감률
2,1,서울,서울,서울,서울,86.72,0.45,86.82,0.12,86.91,...,100.00,1.07,100.74,0.74,101.08,0.34,101.64,0.55,102.71,1.06
3,2,서울,강북지역,도심권,용산구,78.21,0.31,78.32,0.15,78.39,...,100.00,1.37,100.42,0.42,100.21,-0.21,100.11,-0.10,100.66,0.55
4,3,서울,강북지역,동북권,동북권,94.29,0.35,94.55,0.27,94.83,...,100.00,0.84,100.88,0.88,101.47,0.59,102.23,0.74,103.46,1.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,199,경남,밀양시,밀양시,밀양시,96.17,-0.54,95.84,-0.34,95.68,...,100.00,-0.03,100.15,0.15,100.26,0.11,100.34,0.08,100.44,0.10
201,200,경남,거제시,거제시,거제시,129.15,-0.25,128.88,-0.21,128.78,...,100.00,-0.42,99.55,-0.45,99.35,-0.20,99.07,-0.28,98.83,-0.25
202,201,경남,양산시,양산시,양산시,101.91,-0.40,101.67,-0.24,101.40,...,100.00,-0.10,99.99,-0.01,100.15,0.16,100.26,0.11,100.23,-0.03
203,202,제주,제주시,제주시,제주시,92.75,-0.29,92.59,-0.18,92.45,...,100.00,-0.13,99.84,-0.16,99.70,-0.14,99.51,-0.19,99.30,-0.21


In [35]:
# 1. 서울 전체 데이터(2번 행)만 추출
df_target = hpi_df.iloc[[2]].copy()

# 2. 날짜 컬럼 추출: df가 아니라 'hpi_df'에서 찾아야 합니다!
date_cols = [col for col in hpi_df.columns if '년' in col and '.1' not in col]

# 3. melt() 변환: 🚨 전체 데이터(hpi_df)가 아니라, 방금 뽑아낸 'df_target'을 변환해야 합니다!
df_melted = df_target.melt(
    value_vars=date_cols,
    var_name='날짜',
    value_name='housing_price_index' 
)

# 4. '2020년 1월' 형태에서 '년'과 '월'을 분리하여 숫자로 변환
df_melted['년'] = df_melted['날짜'].str.split('년').str[0].astype(int)
df_melted['월'] = df_melted['날짜'].str.split('년').str[1].str.replace('월', '').str.strip().astype(int)

# 5. 지수 데이터를 실수형(float)으로 변환
df_melted['아파트 매매가격지수'] = df_melted['housing_price_index'].astype(float)

# 6. 최종적으로 원하는 컬럼만 선택하고 순서 정렬
hpi = df_melted[['년', '월', '아파트 매매가격지수']]

# 최종 결과 확인
print(hpi)

       년  월  아파트 매매가격지수
0   2020  1       86.72
1   2020  2       86.82
2   2020  3       86.91
3   2020  4       86.82
4   2020  5       86.65
..   ... ..         ...
72  2026  1      100.00
73  2026  2      100.74
74  2026  3      101.08
75  2026  4      101.64
76  2026  5      102.71

[77 rows x 3 columns]


In [36]:
hpi

,년,월,아파트 매매가격지수
0,2020,1,86.72
1,2020,2,86.82
2,2020,3,86.91
3,2020,4,86.82
4,2020,5,86.65
...,...,...,...
72,2026,1,100.00
73,2026,2,100.74
74,2026,3,101.08
75,2026,4,101.64


### 소비자 물가지수 데이터

- [국가데이터처 소비자물가조사 소비자물가지수(2020=100)](https://kosis.kr/statHtml/statHtml.do?orgId=101&tblId=DT_1J22003&conn_path=I2)

In [41]:
cpi_df = pd.read_csv("/Users/CHOIEUNSEO/Desktop/AI·AX 전문가 과정/프로젝트/data/소비자물가지수.csv")

In [42]:
# 1. 날짜 컬럼들만 추출 (이번엔 '.'이 포함된 컬럼명 찾기)
date_cols = [col for col in cpi_df.columns if '.' in str(col)]

# 2. '전국' 데이터가 있는 0번 행만 선택하여 복사
df_target = cpi_df.iloc[[0]].copy()

# 3. melt()를 사용하여 넓은 데이터를 길게 변환
# (이때 id_vars를 지정하지 않으면 '시도별' 컬럼은 자연스럽게 제외됩니다!)
df_melted = df_target.melt(
    value_vars=date_cols,
    var_name='날짜',
    value_name='지수'
)

# 4. '날짜' 컬럼(예: '2020.01')을 '.'을 기준으로 '년'과 '월'로 분리
df_melted['년'] = df_melted['날짜'].str.split('.').str[0].astype(int)
df_melted['월'] = df_melted['날짜'].str.split('.').str[1].astype(int)

# 5. 지수 데이터를 실수형(float)으로 변환
df_melted['소비자물가지수'] = df_melted['지수'].astype(float)

# 6. 최종적으로 원하는 컬럼만 선택하고 순서 정렬
cpi = df_melted[['년', '월', '소비자물가지수']]

## 데이터 통합


In [47]:
from functools import reduce
  
# 1. 병합할 데이터프레임들을 리스트로 묶기
dfs = [trade_cnt, base_rate, mortgage_rate, hpi, cpi]
  
# 2. '년'과 '월'을 기준으로 모든 데이터|프레임 병합 (순차적 left join)        
merged_df = reduce(lambda left, right: pd.merge(left, right, on=['년', '월'],how='left'), dfs)
  
# 3. 2026년 6월 데이터 제외하고 2026년 5월 데이터까지만 맞추기
merged_df = merged_df[~((merged_df['년'] == 2026) & (merged_df['월'] >= 6))].reset_index(drop=True)

In [48]:
merged_df

,년,월,시도,거래량,기준금리,주택담보대출금리,아파트 매매가격지수,소비자물가지수
0,2020,1,서울특별시,6500,1.25,2.51,86.72,100.09
1,2020,2,서울특별시,8403,1.25,2.52,86.82,100.16
2,2020,3,서울특별시,4634,0.75,2.48,86.91,99.94
3,2020,4,서울특별시,3165,0.75,2.58,86.82,99.50
4,2020,5,서울특별시,5819,0.50,2.52,86.65,99.44
...,...,...,...,...,...,...,...,...
72,2026,1,서울특별시,5550,2.50,4.29,100.00,118.03
73,2026,2,서울특별시,5980,2.50,4.32,100.74,118.40
74,2026,3,서울특별시,5702,2.50,4.34,101.08,118.80
75,2026,4,서울특별시,8832,2.50,4.31,101.64,119.37


In [49]:
import os

# 1. 저장할 폴더 경로 설정
save_dir = '/Users/CHOIEUNSEO/Desktop/AI·AX 전문가 과정/프로젝트/data'

# 폴더가 혹시 없다면 생성해주는 코드 (안전장치)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 2. 최종 파일 경로 설정 (원하시는 파일명으로 변경 가능합니다)
file_path = f'{save_dir}/merged_df.csv'

# 3. CSV 파일로 저장
# index=False: 데이터프레임의 인덱스 번호는 엑셀에 저장하지 않음
# encoding='utf-8-sig': 맥(Mac)과 윈도우(Windows) 어디서든 한글이 깨지지 않도록 설정
merged_df.to_csv(file_path, index=False, encoding='utf-8-sig')

print(f"데이터가 성공적으로 저장되었습니다: {file_path}")

데이터가 성공적으로 저장되었습니다: /Users/CHOIEUNSEO/Desktop/AI·AX 전문가 과정/프로젝트/data/merged_df.csv
